# GalaxEye Change Detection — Dataset Exploration

This notebook analyzes the change detection dataset:
- Directory structure & file counts
- Image dimensions & channel counts (EO vs SAR)
- Class distribution before and after label remapping
- Sample visualizations

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from collections import Counter
from tqdm import tqdm
import pandas as pd

DATA_ROOT = Path('../data')
print('Splits:', [d.name for d in DATA_ROOT.iterdir() if d.is_dir() and d.name in ['train','val','test']])

## 1. Directory Structure & File Counts

In [ ]:
summary = []
for split in ['train', 'val', 'test']:
    split_dir = DATA_ROOT / split
    if not split_dir.exists():
        print(f'{split}/ not found'); continue
    subdirs = sorted([d.name for d in split_dir.iterdir() if d.is_dir()])
    for sd in subdirs:
        n = len(list((split_dir / sd).glob('*')))
        summary.append({'split': split, 'folder': sd, 'count': n})

df = pd.DataFrame(summary)
print(df.to_string(index=False))
print(f'\nTotal samples per split:')
for split in ['train','val','test']:
    sub = df[df['split']==split]
    if len(sub) > 0:
        print(f'  {split}: {sub["count"].min()} matched pairs')

## 2. Image Properties (Dimensions, Channels, Dtype)

In [ ]:
def analyze_images(folder, n_samples=20):
    files = sorted(list(folder.glob('*')))[:n_samples]
    shapes, dtypes, channels = [], [], []
    for f in files:
        img = cv2.imread(str(f), cv2.IMREAD_UNCHANGED)
        if img is not None:
            shapes.append(img.shape[:2])
            dtypes.append(str(img.dtype))
            channels.append(img.shape[2] if len(img.shape)==3 else 1)
    return shapes, dtypes, channels

for split in ['train', 'val', 'test']:
    split_dir = DATA_ROOT / split
    if not split_dir.exists(): continue
    print(f'\n=== {split.upper()} ===')
    for folder_name in ['A', 'B', 'OUT']:
        folder = split_dir / folder_name
        if not folder.exists(): continue
        shapes, dtypes, ch = analyze_images(folder)
        unique_shapes = set(shapes)
        unique_ch = set(ch)
        print(f'  {folder_name}/: shapes={unique_shapes}, channels={unique_ch}, dtype={set(dtypes)}')

## 3. Class Distribution (Before & After Remapping)

In [ ]:
REMAP = {0: 0, 1: 0, 2: 1, 3: 1}

for split in ['train', 'val', 'test']:
    mask_dir = DATA_ROOT / split / 'OUT'
    if not mask_dir.exists(): continue
    files = sorted(list(mask_dir.glob('*')))[:200]
    
    original_counts = Counter()
    binary_counts = Counter()
    
    for f in tqdm(files, desc=f'{split}'):
        mask = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        if mask is None: continue
        unique, counts = np.unique(mask, return_counts=True)
        for u, c in zip(unique, counts):
            original_counts[int(u)] += c
            binary_counts[REMAP.get(int(u), 0)] += c
    
    total = sum(original_counts.values())
    print(f'\n{split.upper()} — Original class distribution ({len(files)} samples):')
    for cls in sorted(original_counts.keys()):
        pct = original_counts[cls] / total * 100
        label = {0:'Background', 1:'Intact', 2:'Damaged', 3:'Destroyed'}.get(cls, '?')
        print(f'  Class {cls} ({label}): {original_counts[cls]:>12,} pixels ({pct:.2f}%)')
    
    print(f'  After remapping:')
    for cls in sorted(binary_counts.keys()):
        pct = binary_counts[cls] / total * 100
        label = 'No Change' if cls == 0 else 'Change'
        print(f'  Class {cls} ({label}): {binary_counts[cls]:>12,} pixels ({pct:.2f}%)')

## 4. Sample Visualizations

In [ ]:
def show_samples(split='train', n=5):
    pre_dir = DATA_ROOT / split / 'A'
    post_dir = DATA_ROOT / split / 'B'
    mask_dir = DATA_ROOT / split / 'OUT'
    
    files = sorted(list(pre_dir.glob('*')))[:n]
    cmap = mcolors.ListedColormap(['black','green','orange','red'])
    
    fig, axes = plt.subplots(n, 4, figsize=(16, 4*n))
    if n == 1: axes = axes[np.newaxis, :]
    
    for i, f in enumerate(files):
        stem = f.stem
        pre = cv2.imread(str(pre_dir / f.name), cv2.IMREAD_UNCHANGED)
        post_name = list(post_dir.glob(f'{stem}.*'))
        post = cv2.imread(str(post_name[0]), cv2.IMREAD_UNCHANGED) if post_name else None
        mask_name = list(mask_dir.glob(f'{stem}.*'))
        mask = cv2.imread(str(mask_name[0]), cv2.IMREAD_GRAYSCALE) if mask_name else None
        
        # Binary remap
        binary_mask = np.zeros_like(mask, dtype=np.uint8) if mask is not None else None
        if mask is not None:
            for src, dst in REMAP.items():
                binary_mask[mask == src] = dst
        
        if pre is not None and len(pre.shape)==3: pre = cv2.cvtColor(pre, cv2.COLOR_BGR2RGB)
        if post is not None and len(post.shape)==3: post = cv2.cvtColor(post, cv2.COLOR_BGR2RGB)
        
        if pre is not None:
            axes[i,0].imshow(pre if len(pre.shape)==3 else pre, cmap='gray' if len(pre.shape)==2 else None)
        axes[i,0].set_title('Pre-event (A)')
        axes[i,0].axis('off')
        
        if post is not None:
            axes[i,1].imshow(post if len(post.shape)==3 else post, cmap='gray' if len(post.shape)==2 else None)
        axes[i,1].set_title('Post-event (B)')
        axes[i,1].axis('off')
        
        if mask is not None:
            axes[i,2].imshow(mask, cmap=cmap, vmin=0, vmax=3)
        axes[i,2].set_title('Original Mask')
        axes[i,2].axis('off')
        
        if binary_mask is not None:
            axes[i,3].imshow(binary_mask, cmap=mcolors.ListedColormap(['black','red']), vmin=0, vmax=1)
        axes[i,3].set_title('Binary Mask (Remapped)')
        axes[i,3].axis('off')
    
    plt.suptitle(f'{split.upper()} — Sample Images', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'../outputs/plots/samples_{split}.png', dpi=120, bbox_inches='tight')
    plt.show()

os.makedirs('../outputs/plots', exist_ok=True)
show_samples('train', n=5)

## 5. Class Imbalance Visualization

In [ ]:
# Compute class ratios for the bar chart
split_ratios = {}
for split in ['train', 'val', 'test']:
    mask_dir = DATA_ROOT / split / 'OUT'
    if not mask_dir.exists(): continue
    files = sorted(list(mask_dir.glob('*')))[:100]
    total_px, change_px = 0, 0
    for f in files:
        m = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        if m is None: continue
        binary = np.isin(m, [2, 3]).astype(int)
        total_px += binary.size
        change_px += binary.sum()
    split_ratios[split] = change_px / total_px * 100

fig, ax = plt.subplots(figsize=(8, 5))
splits = list(split_ratios.keys())
ratios = list(split_ratios.values())
bars = ax.bar(splits, ratios, color=['#2196F3','#FF9800','#4CAF50'], edgecolor='black')
ax.set_ylabel('Change Pixels (%)')
ax.set_title('Class Imbalance: % of Change Pixels per Split')
for bar, r in zip(bars, ratios):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{r:.1f}%', ha='center', fontweight='bold')
ax.set_ylim(0, max(ratios)*1.3)
plt.tight_layout()
plt.savefig('../outputs/plots/class_imbalance.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nClass imbalance ratios:', {k: f'{v:.2f}%' for k, v in split_ratios.items()})

## 6. Summary

**Key Findings:**
- Dataset contains pre/post satellite image pairs with 4-class masks
- Remapped to binary: No Change (0,1) vs Change (2,3)
- Significant class imbalance — change pixels are minority
- Need pos_weight in BCE loss to handle imbalance
- Mixed EO (RGB, 3ch) and SAR (grayscale, 1ch) imagery